# 05: Final Data Preparation and Export

In [1]:
import pandas as pd
import numpy as np
import os

DATA_DIR = '../'
OUTPUT_DIR = '../data/processed'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

listings = pd.read_pickle(os.path.join(DATA_DIR, 'listings_cleaned.pkl'))
print(f"Loaded {len(listings)} listings.")

Loaded 94868 listings.


## 1. Type Conversion and Feature Engineering
Ensure all columns are in the correct format and calculate derived features.

In [2]:
num_cols = ['ttm_revenue', 'ttm_occupancy', 'bedrooms', 'guests', 'amenity_count']
for col in num_cols:
    if col in listings.columns:
        listings[col] = pd.to_numeric(listings[col], errors='coerce').fillna(0)

if 'amenity_list' in listings.columns:
    listings['amenities_text'] = listings['amenity_list'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')

if 'ttm_revenue' in listings.columns and len(listings) > 0:
    listings['revenue_tier'] = pd.qcut(listings['ttm_revenue'], 4, labels=['Bronze', 'Silver', 'Gold', 'Platinum'], duplicates='drop')

if 'ttm_occupancy' in listings.columns:
    listings['ttm_occupancy'] = listings['ttm_occupancy'].clip(0, 1)
    listings['occupancy_tier'] = pd.cut(listings['ttm_occupancy'], bins=[0, 0.4, 0.7, 0.9, 1.1], labels=['Low', 'Mid', 'High', 'Optimal'], include_lowest=True)

print("Feature engineering and type conversion complete.")

Feature engineering and type conversion complete.


## 2. Market-Level Aggregation
Create a summary dataset grouped by city/market to identify geographic trends.

In [3]:
agg_cols = ['city', 'listing_type']
available_agg_cols = [c for c in agg_cols if c in listings.columns]

if available_agg_cols:
    # 1. Create market-level summary
    market_summary = listings.groupby(available_agg_cols).agg({
        'listing_id': 'count',
        'ttm_revenue': ['mean', 'median'],
        'ttm_occupancy': 'mean'
    }).reset_index()

    # Flatten columns
    market_summary.columns = ['_'.join(col).strip('_') for col in market_summary.columns.values]
    market_summary.rename(columns={'listing_id_count': 'market_total_listings'}, inplace=True)
    
    # 2. Merge market stats with the main listings dataframe
    # This creates a single comprehensive dataset for Tableau
    final_merged_data = listings.merge(
        market_summary, on=['city', 'listing_type'], how='left'
    )
    
    # Calculate performance index (listing revenue vs market mean)
    if 'ttm_revenue_mean' in final_merged_data.columns:
        final_merged_data['rev_perf_index'] = final_merged_data['ttm_revenue'] / final_merged_data['ttm_revenue_mean']
    
    print(f"Created final merged dataset with {len(final_merged_data)} rows and {len(final_merged_data.columns)} columns.")
else:
    print("Required columns for aggregation not found.")

Created final merged dataset with 94868 rows and 71 columns.


## 3. Final Export
Save the datasets to the 'data' directory for final use.

In [4]:
if 'final_merged_data' in locals():

    export_cols = ['listing_id', 'city', 'listing_type', 'room_type', 'bedrooms', 'guests', 
                   'ttm_revenue', 'ttm_occupancy', 'amenity_count', 'revenue_tier', 'occupancy_tier',
                   'market_total_listings', 'ttm_revenue_mean', 'ttm_revenue_median', 'ttm_occupancy_mean', 'rev_perf_index']
    
    cols_to_use = [c for c in export_cols if c in final_merged_data.columns]
    
    final_output_path = os.path.join(OUTPUT_DIR, 'final_listings_merged.csv')
    final_merged_data[cols_to_use].to_csv(final_output_path, index=False)
    
    print(f"Final merged dataset successfully exported to: {final_output_path}")
else:
    print("Error: final_merged_data not found. Export failed.")

Final merged dataset successfully exported to: ../data/processed/final_listings_merged.csv
